In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np
from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1
from torch import Tensor
import random
from collections import defaultdict
from typing import List, Tuple, Set, Literal

In [3]:
device: Literal['cuda'] | Literal['cpu'] = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [4]:
image_name_to_person_id: dict[str, int] = {}
person_id_to_image_names: dict[int, list[str]] = defaultdict(list)

with open("identity_CelebA.txt", "r") as lines:
    for line in lines:
        image_name, person_id_str = line.split(" ")
        person_id = int(person_id_str.strip())
        image_name_to_person_id[image_name.strip()] = person_id
        person_id_to_image_names[person_id].append(image_name.strip())

person_ids_with_multiple_images: list[int] = [pid for pid, images in person_id_to_image_names.items() if len(images) > 1]
all_image_names = list(image_name_to_person_id.keys())

DATA_DIR = "img_align_celeba/"

In [5]:
def get_balanced_training_data(
    total_pairs: int, 
    exclude_images: Set[str] = set()
) -> List[Tuple[str, str, int]]:
    """
    Generates a balanced dataset of image pairs for training or validation.
    Aims for a 50/50 split between "same person" and "different people" pairs.
    Ensures that no images from the `exclude_images` set are used.
    """
    data = set()
    used_pairs = set()

    # --- 1. Generate "Same Person" (positive) pairs ---
    num_same_pairs = total_pairs // 2
    
    available_persons = [
        pid for pid in person_ids_with_multiple_images 
        if not all(img in exclude_images for img in person_id_to_image_names[pid])
    ]
    if not available_persons:
        raise ValueError("Not enough available persons to generate 'same' pairs after exclusions.")

    while len(data) < num_same_pairs:
        person_id = random.choice(available_persons)
        possible_images = [img for img in person_id_to_image_names[person_id] if img not in exclude_images]
        
        if len(possible_images) < 2:
            continue

        img1, img2 = random.sample(possible_images, 2)
        
        pair = tuple(sorted((img1, img2)))
        if pair not in used_pairs:
            data.add((img1, img2, 1)) # Label 1 for "same"
            used_pairs.add(pair)
            
    # --- 2. Generate "Different People" (negative) pairs ---
    available_images = [img for img in all_image_names if img not in exclude_images]
    if len(available_images) < 2:
        raise ValueError("Not enough available images to generate 'different' pairs after exclusions.")

    while len(data) < total_pairs:
        img1, img2 = random.sample(available_images, 2)

        if image_name_to_person_id[img1] != image_name_to_person_id[img2]:
            pair = tuple(sorted((img1, img2)))
            if pair not in used_pairs:
                data.add((img1, img2, 0)) # Label 0 for "different"
                used_pairs.add(pair)

    final_data = list(data)
    random.shuffle(final_data)
    return final_data

In [6]:
class FaceVerificationMLP(nn.Module):
    '''The MLP model that classifies the difference vector between two face embeddings.'''
    def __init__(self, input_dim=512):
        super(FaceVerificationMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2) # Output 2 logits for binary classification
        )

    def forward(self, x):
        return self.model(x)

# --- Load pre-trained models for face detection and embedding ---
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device=device, post_process=False)
facenet: InceptionResnetV1 = InceptionResnetV1(pretrained='vggface2', device=device).eval()
    
def get_embedding(image_path: str) -> Tensor | None:
    '''Extracts a face embedding from a given image path.'''
    try:
        img = Image.open(DATA_DIR + image_path).convert('RGB')
    except FileNotFoundError:
        print(f"Error: Could not find image {image_path}")
        return None

    return get_embeddings_from_image(img)

def get_embeddings_from_image(image: Image.Image) -> Tensor | None:
    '''Extracts a face embedding from a given PIL Image.'''
    face_tensor = mtcnn(image)
    if face_tensor is None:
        return None # Skip images where no face is detected

    face_tensor = face_tensor.to(device)
    embedding = facenet(face_tensor.unsqueeze(0))
    return embedding.detach()

def get_diff_vector(img1_path: str, img2_path: str) -> Tensor | None:
    '''Calculates the absolute difference vector between two image embeddings.'''
    emb1 = get_embedding(img1_path)
    emb2 = get_embedding(img2_path)
    return get_diff_vector_from_embeddings(emb1, emb2)

def get_diff_vector_from_embeddings(image_1_embeddings: Tensor | None, image_2_embeddings: Tensor | None) -> Tensor | None:
    '''Calculates the absolute difference vector between two image embeddings.'''
    if image_1_embeddings is None or image_2_embeddings is None:
        return None
    return torch.abs(image_1_embeddings - image_2_embeddings)

def augment_image(image, augment_type="gaussian_noise"):
    '''Example augmentation function'''
    if augment_type == "gaussian_noise":
        image_np = np.array(image).astype(np.float32)
        noise = np.random.normal(0, 25, image_np.shape)
        noisy_image = image_np + noise
        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
        augmented = Image.fromarray(noisy_image)
    elif augment_type == "blur":
        augmented = image.filter(ImageFilter.GaussianBlur(radius=3))
    elif augment_type == "increased_lighting":
        enhancer = ImageEnhance.Brightness(image)
        augmented = enhancer.enhance(1.5)
    else:
        augmented = image
    return augmented

In [7]:
from torch._tensor import Tensor


def generate_embeddings_for_pairs(pairs: List[Tuple[str, str, int]]):    
    X_train, y_train = [], []
    print("Generating embeddings for training data...")
    for i, (img1, img2, label) in enumerate(pairs):
        print(f"  Processing pair {i+1}/{len(pairs)}...", end='\r')
        diff = get_diff_vector(img1, img2)
        if diff is not None:
            X_train.append(diff.squeeze(0))
            y_train.append(label)
            
    return X_train, y_train

def train_model(
    X_train,
    y_train,
    lr: float,
    epochs: int,
    batch_size: int = 64
) -> FaceVerificationMLP:
    """
    Trains an MLP model from scratch given training data and hyperparameters.
    """

    if not X_train:
        raise ValueError("Could not generate any valid training embeddings.")

    X_train_tensor = torch.stack(X_train).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)

    model = FaceVerificationMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    print(f"Starting model training: {epochs} epochs, lr={lr}, batch_size={batch_size}")
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(X_train_tensor.size()[0])
        
        for i in range(0, X_train_tensor.size()[0], batch_size):
            optimizer.zero_grad()
            indices = permutation[i:i+batch_size]
            batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            print(f"  Epoch {epoch+1}/{epochs} completed.")
            
    return model

def get_diffs(
    pairs: List[Tuple[str, str, int]],
    augment_type: str|None=None
) -> List[Tuple[Tensor, int]]:
    """
    Generates difference vectors for a list of image pairs.
    Optionally applies augmentation to the images.
    """
    diffs = []
    print("Generating difference vectors...")
    
    for i, (img1, img2, label) in enumerate(pairs):
        print(f"  Processing pair {i+1}/{len(pairs)}...", end='\r')
        if augment_type is not None:
            img1 = augment_image(Image.open(DATA_DIR + img1), augment_type)
            img2 = augment_image(Image.open(DATA_DIR + img2), augment_type)
            diff = get_diff_vector_from_embeddings(
                get_embeddings_from_image(img1), get_embeddings_from_image(img2)
            )
        else:
            diff = get_diff_vector(img1, img2)
        
        if diff is not None:
            diffs.append((diff, label))
    
    return diffs

def evaluate_model(model: nn.Module, diffs: List[Tuple[Tensor, int]], augment_type: str|None=None) -> dict:
    """
    Evaluates a trained model on a validation set and returns performance metrics.
    """
    print("Evaluating model...")
    tp, fp, tn, fn = 0, 0, 0, 0
    model.eval()
    
    with torch.no_grad():
        for i, (diff, true_label) in enumerate(diffs):
            print(f"  Evaluating pair {i+1}/{len(diffs)}", end='\r')
            
            diff = diff.to(device)
            output = model(diff)
            _, predicted_label = torch.max(output, 1)
            predicted_label = predicted_label.item()

            if predicted_label == 1 and true_label == 1: tp += 1
            elif predicted_label == 1 and true_label == 0: fp += 1
            elif predicted_label == 0 and true_label == 0: tn += 1
            elif predicted_label == 0 and true_label == 1: fn += 1
            
    print("\nEvaluation complete.")
    
    total = tp + fp + tn + fn
    epsilon = 1e-6
    accuracy = (tp + tn) / (total + epsilon)
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f1_score = 2 * (precision * recall) / (precision + recall + epsilon)
    
    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1_score,
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn
    }

Zadanie 1: Analiza wielkości training set size na wyniki modelu

In [8]:
train_sizes = [10, 100, 300, 500, 700, 1000, 5000]
VALIDATION_SET_SIZE = 200
FIXED_LR = 0.001
FIXED_EPOCHS = 20
results_task1 = {}

for n in train_sizes:
    print("-" * 60)
    print(f"STARTING EXPERIMENT WITH N = {n} TRAINING PAIRS")
    
    print("\nStep 1: Generating data...")
    train_data = get_balanced_training_data(n)
    exclude_images = {img for pair in train_data for img in pair[:2]}
    validation_data = get_balanced_training_data(VALIDATION_SET_SIZE, exclude_images=exclude_images)
    print(f"Generated {len(train_data)} training pairs and {len(validation_data)} validation pairs.")
    
    X_train, y_train = generate_embeddings_for_pairs(train_data)

    validation_diffs = get_diffs(validation_data)
    
    print("\nStep 2: Training model...")
    model = train_model(X_train, y_train, lr=FIXED_LR, epochs=FIXED_EPOCHS)
        
    print("\nStep 3: Evaluating model...")
    metrics = evaluate_model(model, validation_diffs)
    results_task1[n] = metrics
    
    print("\n--- Metrics for N =", n, "---")
    print(f"  Accuracy:  {metrics['accuracy']:.2%}")
    print(f"  Precision: {metrics['precision']:.2%}")
    print(f"  Recall:    {metrics['recall']:.2%}")
    print(f"  F1-Score:  {metrics['f1_score']:.2%}")
    print(f"  (TP: {metrics['tp']}, FP: {metrics['fp']}, TN: {metrics['tn']}, FN: {metrics['fn']})")

print("\n" + "="*70)
print(" " * 15 + "TASK 1: FINAL RESULTS SUMMARY (TRAINING SET SIZE)" + " " * 15)
print("="*70)
print(f"{'N (Train Pairs)':>16} | {'Accuracy':>12} | {'Precision':>12} | {'Recall':>12} | {'F1-Score':>12}")
print("-" * 70)
for n, metrics in results_task1.items():
    print(f"{n:>16} | {metrics['accuracy']:>12.2%} | {metrics['precision']:>12.2%} | {metrics['recall']:>12.2%} | {metrics['f1_score']:>12.2%}")
print("="*70)

------------------------------------------------------------
STARTING EXPERIMENT WITH N = 10 TRAINING PAIRS

Step 1: Generating data...
Generated 10 training pairs and 200 validation pairs.
Generating embeddings for training data...
Generating difference vectors...
  Processing pair 200/200...
Step 2: Training model...
Starting model training: 20 epochs, lr=0.001, batch_size=64
  Epoch 5/20 completed.
  Epoch 10/20 completed.
  Epoch 15/20 completed.
  Epoch 20/20 completed.

Step 3: Evaluating model...
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

--- Metrics for N = 10 ---
  Accuracy:  54.77%
  Precision: 53.47%
  Recall:    77.00%
  F1-Score:  63.11%
  (TP: 77, FP: 67, TN: 32, FN: 23)
------------------------------------------------------------
STARTING EXPERIMENT WITH N = 100 TRAINING PAIRS

Step 1: Generating data...
Generated 100 training pairs and 200 validation pairs.
Generating embeddings for training data...
Generating difference vectors...
  Processing 

Co ciekawe, najwyższy F1-Score ma model wytrenowany na N=700. Wysoki wynik osiągnął również N=500 i N=5000.

Najwyższe accuracy było dla N=5000, potem N=700 i N=500.

In [9]:
fixed_train_data_lr: List[Tuple[str, str, int]] = get_balanced_training_data(1000)
exclude_images_lr: set[str] = {img for pair in fixed_train_data_lr for img in pair[:2]}
fixed_validation_data_lr = get_balanced_training_data(VALIDATION_SET_SIZE, exclude_images=exclude_images_lr)
print(f"Generated fixed dataset: {len(fixed_train_data_lr)} training pairs, {len(fixed_validation_data_lr)} validation pairs.\n")

X_train, y_train = generate_embeddings_for_pairs(fixed_train_data_lr)

validation_diffs = get_diffs(fixed_validation_data_lr)

Generated fixed dataset: 1000 training pairs, 200 validation pairs.

Generating embeddings for training data...
Generating difference vectors...


Zadanie 2: testy learning rate

In [10]:
FIXED_EPOCHS_FOR_LR = 20
learning_rates = [1e-6, 1e-5, 1e-4, 5e-4, 8e-4, 1e-3, 2e-3, 3e-3, 4e-3, 5e-3, 1e-2, 1e-1, 1]
results_task2 = {}

for lr in learning_rates:
    print("-" * 60)
    print(f"STARTING EXPERIMENT WITH LEARNING RATE = {lr}")
    
    model = train_model(X_train, y_train, lr=lr, epochs=FIXED_EPOCHS_FOR_LR)
    
    metrics = evaluate_model(model, validation_diffs)
    results_task2[lr] = metrics
    
    print("\n--- Metrics for LR =", lr, "---")
    print(f"  Accuracy:  {metrics['accuracy']:.2%}")
    print(f"  F1-Score:  {metrics['f1_score']:.2%}")

print("\n" + "="*70)
print(" " * 17 + "TASK 2: FINAL RESULTS SUMMARY (LEARNING RATE)" + " " * 18)
print("="*70)
print(f"{'Learning Rate':>16} | {'Accuracy':>12} | {'Precision':>12} | {'Recall':>12} | {'F1-Score':>12}")
print("-" * 70)
for lr, metrics in results_task2.items():
    print(f"{lr:>16.5f} | {metrics['accuracy']:>12.2%} | {metrics['precision']:>12.2%} | {metrics['recall']:>12.2%} | {metrics['f1_score']:>12.2%}")
print("="*70)

------------------------------------------------------------
STARTING EXPERIMENT WITH LEARNING RATE = 1e-06
Starting model training: 20 epochs, lr=1e-06, batch_size=64
  Epoch 5/20 completed.
  Epoch 10/20 completed.
  Epoch 15/20 completed.
  Epoch 20/20 completed.
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

--- Metrics for LR = 1e-06 ---
  Accuracy:  49.75%
  F1-Score:  66.44%
------------------------------------------------------------
STARTING EXPERIMENT WITH LEARNING RATE = 1e-05
Starting model training: 20 epochs, lr=1e-05, batch_size=64
  Epoch 5/20 completed.
  Epoch 10/20 completed.
  Epoch 15/20 completed.
  Epoch 20/20 completed.
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

--- Metrics for LR = 1e-05 ---
  Accuracy:  50.25%
  F1-Score:  0.00%
------------------------------------------------------------
STARTING EXPERIMENT WITH LEARNING RATE = 0.0001
Starting model training: 20 epochs, lr=0.0001, batch_size=64
  Epoch 5/20 comple

Jeśli celem jest najlepszy ogólny balans, to najlepszy jest lr = 0.00300, ponieważ osiąga najwyższy F1-Score (64.31%). Trzeba jednak pamiętać o jego skłonności do wysokiej czułości i niższej precyzji.
Bardzo dobrym, bezpieczniejszym kandydatem jest lr = 0.00080. Ma on drugi najwyższy F1-Score (62.88%), ale jego precyzja (55.38%) i czułość (72.73%) są znacznie lepiej zrównoważone niż w przypadku lr = 0.00300.

Model, co najrzadziej będzie popełniał błąd "false positive" jest model o lr = 0.00010 z najwyższą precyzją (68.42%).

Model, który wykryje najwięcej prawdziwych par jest model o lr = 0.00300 z najwyższą czułością (82.83%).

Zadanie 3: epochs

In [14]:
FIXED_LR_FOR_EPOCH = 1e-3
EPOCH_COUNT = [5, 7, 9, 11, 13, 15, 17, 20, 25, 30, 50]
results_task3 = {}

for epochs in EPOCH_COUNT:
    print("-" * 60)
    print(f"STARTING EXPERIMENT WITH EPOCH COUNT = {epochs}")
    
    model = train_model(X_train, y_train, lr=FIXED_LR_FOR_EPOCH, epochs=epochs)
    
    metrics = evaluate_model(model, validation_diffs)
    results_task3[epochs] = metrics
    
    print("\n--- Metrics for Epochs =", epochs, "---")
    print(f"  Accuracy:  {metrics['accuracy']:.2%}")
    print(f"  F1-Score:  {metrics['f1_score']:.2%}")

print("\n" + "="*70)
print(" " * 17 + "TASK 3: FINAL RESULTS SUMMARY (NUMBER OF EPOCHS)" + " " * 16)
print("="*70)       
print(f"{'Num. of Epochs':>16} | {'Accuracy':>12} | {'Precision':>12} | {'Recall':>12} | {'F1-Score':>12}")
print("-" * 70)     
for epochs, metrics in results_task3.items():
    print(f"{epochs:>16} | {metrics['accuracy']:>12.2%} | {metrics['precision']:>12.2%} | {metrics['recall']:>12.2%} | {metrics['f1_score']:>12.2%}")
print("="*70)

------------------------------------------------------------
STARTING EXPERIMENT WITH EPOCH COUNT = 5
Starting model training: 5 epochs, lr=0.001, batch_size=64
  Epoch 5/5 completed.
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

--- Metrics for Epochs = 5 ---
  Accuracy:  58.29%
  F1-Score:  54.14%
------------------------------------------------------------
STARTING EXPERIMENT WITH EPOCH COUNT = 7
Starting model training: 7 epochs, lr=0.001, batch_size=64
  Epoch 5/7 completed.
  Epoch 7/7 completed.
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

--- Metrics for Epochs = 7 ---
  Accuracy:  56.78%
  F1-Score:  60.91%
------------------------------------------------------------
STARTING EXPERIMENT WITH EPOCH COUNT = 9
Starting model training: 9 epochs, lr=0.001, batch_size=64
  Epoch 5/9 completed.
  Epoch 9/9 completed.
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

--- Metrics for Epochs = 9 ---
  Accuracy:  55.28%
  F1-

Najlepsze F1-Score osiągnięty był przy liczbie epochs równej 25 - osiągnął wysokie wartości we wszystkich metrykach

Zadanie 4: najlepsze hiperparametry

In [31]:
FIXED_LR_FOR_EPOCH = 3e-3
EPOCH_COUNT = 25

model = train_model(X_train, y_train, lr=FIXED_LR_FOR_EPOCH, epochs=EPOCH_COUNT)

metrics = evaluate_model(model, validation_diffs)

print("\n" + "="*70)
print(" " * 17 + "TASK 4: FINAL RESULTS SUMMARY" + " " * 16)
print("="*70)
print(f"{metrics['accuracy']:>12.2%} | {metrics['precision']:>12.2%} | {metrics['recall']:>12.2%} | {metrics['f1_score']:>12.2%}")

Starting model training: 25 epochs, lr=0.003, batch_size=64
  Epoch 5/25 completed.
  Epoch 10/25 completed.
  Epoch 15/25 completed.
  Epoch 20/25 completed.
  Epoch 25/25 completed.
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

                 TASK 4: FINAL RESULTS SUMMARY                
      55.78% |       53.74% |       79.80% |       64.23%


Zadanie 5: Analiza odporności na zakłócenia i strategia mitygacji

In [32]:
OPTIMAL_N = 700
OPTIMAL_LR = 0.003
OPTIMAL_EPOCHS = 25
VALIDATION_SET_SIZE = 200

print("Generating fixed datasets for experiments...")
fixed_train_data = get_balanced_training_data(OPTIMAL_N)
exclude_images = {img for pair in fixed_train_data for img in pair[:2]}
fixed_validation_data = get_balanced_training_data(VALIDATION_SET_SIZE, exclude_images=exclude_images)
print(f"Generated fixed dataset: {len(fixed_train_data)} training pairs, {len(fixed_validation_data)} validation pairs.\n")

Generating fixed datasets for experiments...
Generated fixed dataset: 700 training pairs, 200 validation pairs.



#### Krok 1: Wydajność bazowa - Model trenowany na czystych danych

In [33]:
print("-" * 60)
print("PART 1: TRAINING BASELINE MODEL ON CLEAN DATA")
print("-" * 60)

# Generowanie embeddingów dla czystego zbioru treningowego
X_train_vanilla, y_train_vanilla = generate_embeddings_for_pairs(fixed_train_data)

# Trenowanie modelu "vanilla"
vanilla_model = train_model(X_train_vanilla, y_train_vanilla, lr=OPTIMAL_LR, epochs=OPTIMAL_EPOCHS)

------------------------------------------------------------
PART 1: TRAINING BASELINE MODEL ON CLEAN DATA
------------------------------------------------------------
Generating embeddings for training data...
Starting model training: 25 epochs, lr=0.003, batch_size=64
  Epoch 5/25 completed.
  Epoch 10/25 completed.
  Epoch 15/25 completed.
  Epoch 20/25 completed.
  Epoch 25/25 completed.


In [34]:
# Przygotowanie 4 wariantów zbioru walidacyjnego (oryginalny + 3 z augmentacjami)
print("\nGenerating difference vectors for all validation sets...")
validation_diffs_original = get_diffs(fixed_validation_data, augment_type=None)
validation_diffs_noise = get_diffs(fixed_validation_data, augment_type="gaussian_noise")
validation_diffs_blur = get_diffs(fixed_validation_data, augment_type="blur")
validation_diffs_light = get_diffs(fixed_validation_data, augment_type="increased_lighting")

# Ewaluacja modelu "vanilla" na wszystkich 4 zestawach
results_vanilla = {}
print("\n" + "="*70)
print("EVALUATING BASELINE MODEL (TRAINED ON CLEAN DATA)")
print("="*70)

print("\nEvaluating on ORIGINAL validation data...")
results_vanilla['Original'] = evaluate_model(vanilla_model, validation_diffs_original)

print("\nEvaluating on GAUSSIAN NOISE validation data...")
results_vanilla['Gaussian Noise'] = evaluate_model(vanilla_model, validation_diffs_noise)

print("\nEvaluating on BLUR validation data...")
results_vanilla['Blur'] = evaluate_model(vanilla_model, validation_diffs_blur)

print("\nEvaluating on INCREASED LIGHTING validation data...")
results_vanilla['Increased Lighting'] = evaluate_model(vanilla_model, validation_diffs_light)

print("\n" + "="*70)
print(" " * 10 + "BASELINE MODEL PERFORMANCE SUMMARY" + " " * 10)
print("="*70)
print(f"{'Validation Set':<20} | {'Accuracy':>12} | {'Precision':>12} | {'Recall':>12} | {'F1-Score':>12}")
print("-" * 70)
for name, metrics in results_vanilla.items():
    print(f"{name:<20} | {metrics['accuracy']:>12.2%} | {metrics['precision']:>12.2%} | {metrics['recall']:>12.2%} | {metrics['f1_score']:>12.2%}")
print("="*70)


Generating difference vectors for all validation sets...
Generating difference vectors...
Generating difference vectors...
Generating difference vectors...
Generating difference vectors...
  Processing pair 200/200...
EVALUATING BASELINE MODEL (TRAINED ON CLEAN DATA)

Evaluating on ORIGINAL validation data...
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

Evaluating on GAUSSIAN NOISE validation data...
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

Evaluating on BLUR validation data...
Evaluating model...
  Evaluating pair 198/198
Evaluation complete.

Evaluating on INCREASED LIGHTING validation data...
Evaluating model...
  Evaluating pair 187/187
Evaluation complete.

          BASELINE MODEL PERFORMANCE SUMMARY          
Validation Set       |     Accuracy |    Precision |       Recall |     F1-Score
----------------------------------------------------------------------
Original             |       56.78% |       56.07% |       60.61% |    

In [35]:
def generate_augmented_embeddings(
    pairs: List[Tuple[str, str, int]], 
    augment_types: List[str]
) -> Tuple[List[Tensor], List[int]]:
    """
    Generates difference vectors for training, applying a random augmentation 
    from the provided list to each image in a pair.
    """
    X_train, y_train = [], []
    print("Generating augmented embeddings for training data...")
    
    for i, (img1_path, img2_path, label) in enumerate(pairs):
        print(f"  Processing pair {i+1}/{len(pairs)}...", end='\r')
        
        try:
            # Wczytaj obrazy
            img1_pil = Image.open(DATA_DIR + img1_path).convert('RGB')
            img2_pil = Image.open(DATA_DIR + img2_path).convert('RGB')

            # Zastosuj losową augmentację do każdego obrazu
            aug_type1 = random.choice(augment_types)
            aug_type2 = random.choice(augment_types)
            
            aug_img1 = augment_image(img1_pil, aug_type1)
            aug_img2 = augment_image(img2_pil, aug_type2)
            
            # Wygeneruj wektor różnic
            diff = get_diff_vector_from_embeddings(
                get_embeddings_from_image(aug_img1),
                get_embeddings_from_image(aug_img2)
            )
            
            if diff is not None:
                X_train.append(diff.squeeze(0))
                y_train.append(label)
        except FileNotFoundError:
            continue
            
    print("\nGeneration of augmented embeddings complete.")
    return X_train, y_train

# %%
print("\n" + "-" * 60)
print("PART 2: TRAINING AUGMENTED MODEL")
print("-" * 60)

# Lista augmentacji do użycia podczas treningu
augment_list = ["gaussian_noise", "blur", "increased_lighting", "none"] 

# Generowanie ZAAUGMENTOWANEGO zbioru treningowego
X_train_aug, y_train_aug = generate_augmented_embeddings(fixed_train_data, augment_types=augment_list)

# Trenowanie modelu "augmented"
augmented_model = train_model(X_train_aug, y_train_aug, lr=OPTIMAL_LR, epochs=OPTIMAL_EPOCHS)


------------------------------------------------------------
PART 2: TRAINING AUGMENTED MODEL
------------------------------------------------------------
Generating augmented embeddings for training data...
  Processing pair 700/700...
Generation of augmented embeddings complete.
Starting model training: 25 epochs, lr=0.003, batch_size=64
  Epoch 5/25 completed.
  Epoch 10/25 completed.
  Epoch 15/25 completed.
  Epoch 20/25 completed.
  Epoch 25/25 completed.


In [40]:
# Ewaluacja modelu "augmented" na tych samych 4 zestawach walidacyjnych
results_augmented = {}
print("\n" + "="*70)
print("EVALUATING AUGMENTED MODEL (TRAINED ON AUGMENTED DATA)")
print("="*70)

print("\nEvaluating on ORIGINAL validation data...")
results_augmented['Original'] = evaluate_model(augmented_model, validation_diffs_original)

print("\nEvaluating on GAUSSIAN NOISE validation data...")
results_augmented['Gaussian Noise'] = evaluate_model(augmented_model, validation_diffs_noise)

print("\nEvaluating on BLUR validation data...")
results_augmented['Blur'] = evaluate_model(augmented_model, validation_diffs_blur)

print("\nEvaluating on INCREASED LIGHTING validation data...")
results_augmented['Increased Lighting'] = evaluate_model(augmented_model, validation_diffs_light)

print("\n" + "="*70)
print(" " * 10 + "AUGMENTED MODEL PERFORMANCE SUMMARY" + " " * 10)
print("="*70)
print(f"{'Validation Set':<20} | {'Accuracy':>12} | {'Precision':>12} | {'Recall':>12} | {'F1-Score':>12}")
print("-" * 70)
for name, metrics in results_augmented.items():
    print(f"{name:<20} | {metrics['accuracy']:>12.2%} | {metrics['precision']:>12.2%} | {metrics['recall']:>12.2%} | {metrics['f1_score']:>12.2%}")
print("="*70)


EVALUATING AUGMENTED MODEL (TRAINED ON AUGMENTED DATA)

Evaluating on ORIGINAL validation data...
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

Evaluating on GAUSSIAN NOISE validation data...
Evaluating model...
  Evaluating pair 199/199
Evaluation complete.

Evaluating on BLUR validation data...
Evaluating model...
  Evaluating pair 198/198
Evaluation complete.

Evaluating on INCREASED LIGHTING validation data...
Evaluating model...
  Evaluating pair 187/187
Evaluation complete.

          AUGMENTED MODEL PERFORMANCE SUMMARY          
Validation Set       |     Accuracy |    Precision |       Recall |     F1-Score
----------------------------------------------------------------------
Original             |       54.27% |       55.13% |       43.43% |       48.59%
Gaussian Noise       |       58.79% |       61.04% |       47.47% |       53.41%
Blur                 |       49.49% |       48.15% |       26.53% |       34.21%
Increased Lighting   |       58.82% |   

In [41]:
print("\n" + "="*80)
print(" " * 18 + "FINAL COMPARISON: VANILLA vs. AUGMENTED MODEL" + " " * 18)
print("="*80)
print(f"{'Validation Set':<20} | {'Vanilla Model F1':>20} | {'Augmented Model F1':>22} | {'Improvement':>15}")
print("-" * 80)

for name in results_vanilla.keys():
    f1_vanilla = results_vanilla[name]['f1_score']
    f1_augmented = results_augmented[name]['f1_score']
    improvement = f1_augmented - f1_vanilla
    print(f"{name:<20} | {f1_vanilla:>19.2%} | {f1_augmented:>21.2%} | {improvement:>+14.2%}")

print("="*80)


                  FINAL COMPARISON: VANILLA vs. AUGMENTED MODEL                  
Validation Set       |     Vanilla Model F1 |     Augmented Model F1 |     Improvement
--------------------------------------------------------------------------------
Original             |              58.25% |                48.59% |         -9.66%
Gaussian Noise       |              56.86% |                53.41% |         -3.45%
Blur                 |              32.59% |                34.21% |         +1.62%
Increased Lighting   |              56.04% |                52.76% |         -3.28%


Próba poprawek przyniosła tylko lekką poprawę w tylko jednej kategorii - blur, w pozostałych skomplikowanie danych treningowych wpłynęło negatywnie na wyniki.